In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import time

In [ ]:
start_notebook = time.time()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!nvidia-smi

Thu Jan 15 12:39:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             74W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 1. Load Environment

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

In [ ]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl
import torch

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)
print("torch:", torch.__version__)

transformers: 4.57.3
datasets: 4.0.0
accelerate: 1.12.0
peft: 0.18.0
bitsandbytes: 0.49.1
trl: 0.26.2
torch: 2.9.0+cu126


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 2. Import Libraries

In [ ]:
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# 3. Parameters

In [ ]:
import random
from transformers import set_seed

In [ ]:
name_model = "roberta-base"

In [ ]:
n_total_epochs = 4

In [ ]:
seed = 2

In [ ]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
set_seed(seed)

In [ ]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

# 4. Dataset: DB Pedia

In [ ]:
name_dataset = 'DB_Pedia'

In [ ]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/01.Datasets_Creation/{name_dataset}'

In [ ]:
tokenized_train = load_from_disk(f'{path_open}/{name_model}/train/')

In [ ]:
tokenized_val = load_from_disk(f'{path_open}/{name_model}/val/')

In [ ]:
tokenized_test = load_from_disk(f'{path_open}/{name_model}/test/')

# 5. Load Model and Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(name_model, num_labels = 14)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

# 6. Training arguments

In [ ]:
num_epochs = 1

In [ ]:
training_args = TrainingArguments(
    output_dir = "./results",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = num_epochs,
    weight_decay = 0.01,
    report_to="none",
    save_steps = 0,
    logging_steps = 25,
    logging_strategy = "no",
    disable_tqdm = True,
    lr_scheduler_type = "linear",
    optim="adamw_torch",
    seed=seed,
    data_seed=seed,
    dataloader_num_workers=0
    )

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    data_collator = data_collator
    )

# 7. Train

In [ ]:
y_true_train = tokenized_train["label"]
y_true_val = tokenized_val["label"]

In [ ]:
y_true_train = np.array(y_true_train)
y_true_val = np.array(y_true_val)

In [ ]:
for n_e in range(n_total_epochs):

  start_e = time.time()

  print('\n')
  print(f'Epoch: {n_e + 1}')
  print('-'*70)
  print('\n')

  #--------------------------------------------------

  start = time.time()

  trainer.train()

  end = time.time()

  delta = end - start
  hours, rem = divmod(delta, 3600)
  minutes, seconds = divmod(rem, 60)

  print('\n')
  print(f"Training Time: {int(hours)}h {int(minutes)}m {seconds:.2f}s")
  print('-'*70)
  print('\n')

  #--------------------------------------------------

  pred_train = trainer.predict(tokenized_train)
  pred_val = trainer.predict(tokenized_val)

  y_pred_train = np.argmax(pred_train.predictions, axis=-1)
  y_pred_val = np.argmax(pred_val.predictions, axis=-1)

  #--------------------------------------------------

  print('-'*70)
  print('\n')
  print("📊 Classification Report: Train")
  print('\n')

  f1_train = f1_score(y_true_train, y_pred_train, average='macro')
  accuracy_train = accuracy_score(y_true_train, y_pred_train)
  precision_train = precision_score(y_true_train, y_pred_train, average='macro')
  recall_train = recall_score(y_true_train, y_pred_train, average='macro')

  f1_train = 100*f1_train
  accuracy_train = 100*accuracy_train
  precision_train = 100*precision_train
  recall_train = 100*recall_train

  print('\n')
  print(f"F1 Score:      {f1_train:.2f}")
  print(f"Accuracy:      {accuracy_train:.2f}")
  print(f"Precision:     {precision_train:.2f}")
  print(f"Recall:        {recall_train:.2f}")
  print('\n')
  print('-'*70)

  #--------------------------------------------------

  print('-'*70)
  print('\n')
  print("📊 Classification Report: Validation")
  print('\n')

  f1_val = f1_score(y_true_val, y_pred_val, average='macro')
  accuracy_val = accuracy_score(y_true_val, y_pred_val)
  precision_val = precision_score(y_true_val, y_pred_val, average='macro')
  recall_val = recall_score(y_true_val, y_pred_val, average='macro')

  f1_val = 100*f1_val
  accuracy_val = 100*accuracy_val
  precision_val = 100*precision_val
  recall_val = 100*recall_val

  print('\n')
  print(f"F1 Score:      {f1_val:.2f}")
  print(f"Accuracy:      {accuracy_val:.2f}")
  print(f"Precision:     {precision_val:.2f}")
  print(f"Recall:        {recall_val:.2f}")
  print('\n')
  print('-'*70)

  #--------------------------------------------------

  path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/02.Finetunning_Encoders/{name_dataset}/{name_model}/seed={seed}/epoch_{str(int(n_e + 1))}'

  trainer.save_model(path_save)
  tokenizer.save_pretrained(path_save)

  #--------------------------------------------------

  end_e = time.time()

  delta_e = end_e - start_e
  hours_e, rem_e = divmod(delta_e, 3600)
  minutes_e, seconds_e = divmod(rem_e, 60)

  print('\n')
  print(f"Epoch Execution: {int(hours_e)}h {int(minutes_e)}m {seconds_e:.2f}s")
  print('\n')
  print('-'*70)



Epoch: 1
----------------------------------------------------------------------


{'train_runtime': 3168.1602, 'train_samples_per_second': 176.759, 'train_steps_per_second': 11.047, 'train_loss': 0.053781940569196426, 'epoch': 1.0}


Training Time: 0h 52m 48.65s
----------------------------------------------------------------------


----------------------------------------------------------------------


📊 Classification Report: Train




F1 Score:      99.44
Accuracy:      99.44
Precision:     99.44
Recall:        99.44


----------------------------------------------------------------------
----------------------------------------------------------------------


📊 Classification Report: Validation




F1 Score:      99.26
Accuracy:      99.26
Precision:     99.26
Recall:        99.26


----------------------------------------------------------------------


Epoch Execution: 1h 11m 53.91s


----------------------------------------------------------------------


Epoch: 2
----------

# 8. Execution time

In [ ]:
end_notebook = time.time()

In [ ]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 4h 47m 57.65s
